# 19 — Advanced Review and Capstone

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/19', {1: 'cbf8e9218a372ee8', 2: 'c7fab96aebe7845a', 3: '47e24bef853eaa9d', 4: '06063b05bf77585a', 5: 'ea76bd63fe6186b6', 6: 'ba851c588b560a0b'}, independent='replay_summary')

## 1. Design

- Rules in pure functions; I/O behind ports; dependencies (clock, storage, network) injected.
- Validate at the boundary, raise domain errors, and keep data immutable where you can.
- Use the data model (`__eq__`, `__hash__`, `__repr__`) so objects behave like built-ins.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Money:
    cents: int
    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.cents + other.cents)

print(Money(150) + Money(75), {Money(1), Money(1)})

**Exercise 1.** A frozen dataclass: can you change `cents` after creating it? Write `True` or `False`.

In [ ]:
answer = ___
check(1, answer)

**Exercise 2.** Two equal frozen dataclasses in a set: how many items?

In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class P:
    x: int
predicted = ___   # len({P(1), P(1), P(2)})
check(2, predicted)

## 2. Reliability

- Every external call gets a timeout, a bounded retry for transient errors, and an idempotency key if it changes state.
- Limit concurrency (semaphores) and rate (token buckets).
- Database rules are enforced by constraints and transactions, not by checks in Python.

In [ ]:
delays = [0.1 * 2 ** n for n in range(4)]
print(delays, sum(delays))

**Exercise 3.** Total wait across 3 retries with base 0.2 s and doubling.

In [ ]:
total = round(sum(0.2 * 2 ** n for n in range(___)), 2)
check(3, total)

**Exercise 4.** Which tool caps how many coroutines run at once? Write `asyncio.` plus its name.

In [ ]:
answer = "asyncio." + ___
check(4, answer)

## 3. Operations

- Emit structured events with correlation ids; track p95 latency and availability against an SLO.
- Version APIs; never break v1. Hide 5xx internals.
- Migrations are ordered, transactional and repeatable; config comes from the environment.

In [ ]:
import math
samples = sorted([20, 22, 19, 400, 21, 23, 20, 18, 25, 24])
print("p95:", samples[math.ceil(0.95 * len(samples)) - 1])

**Exercise 5.** Error budget for 99.5% over 2,000 requests.

In [ ]:
import math
budget = math.floor(2000 * (100 - 99.5) / 100)
predicted = ___
check(5, predicted)

**Exercise 6.** Is client 1.4.0 compatible with server 2.0.0? Write `True` or `False`.

In [ ]:
answer = ___
check(6, answer)

## Independent build

Order Desk needs a replay audit. Implement replay_summary(events) for a one-pass stream of (key, amount_in_cents) pairs. Return (accepted_cents, duplicates, conflicts). Only the first delivery per key contributes money. A later matching amount counts as a duplicate; a different amount counts as a conflict and leaves the original amount in effect. Explain what state must survive a restart.

**Delayed revision.** A conflicting replay arrives between two valid deliveries of the same key. Record how finance can distinguish the conflict from the duplicate in your result.

In [ ]:
# Write your implementation here.

In [ ]:
def acceptance_cases(submission):
    assert submission(iter([('A', 5), ('B', 0), ('A', 6), ('A', 5), ('B', 0)])) == (5, 2, 1), "Check the result for replay_summary(iter([('A', 5), ('B', 0), ('A', 6), ('A', 5), ('B', 0)]))"
    assert submission(iter([])) == (0, 0, 0), 'Check the result for replay_summary(iter([]))'

check.assess('replay_summary', acceptance_cases)

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Bounded concurrency means…

   - **a)** A cap on how much work runs at once
   - **b)** No limit on tasks
   - **c)** One task at a time always

**2.** Before optimizing, you should…

   - **a)** Measure
   - **b)** Rewrite in C
   - **c)** Add caching everywhere

**3.** A resilient service handles duplicate requests via…

   - **a)** Restarting
   - **b)** Idempotency
   - **c)** Ignoring them

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/19', {1: 'b9ec0359f1ca', 2: '9897f1ab70c5', 3: 'b25212ffb604'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** the [advanced projects](../projects/) and [exercise workbooks](../exercises/)